
# NeoLM — HellaSwag Benchmark

Evaluate commonsense text continuation using the [HellaSwag](https://huggingface.co/datasets/Rowan/hellaswag) benchmark.

## Evaluation

- Task: 4-choice sentence continuation
- Method: Zero-shot conditional log-likelihood
- Metrics: Accuracy and length-normalized accuracy
- Random chance baseline: 25%

This benchmark evaluates continuation preferences without instruction prompting or fine-tuning.

**Note:** NeoLM-Small-Base is a 61M-parameter base model pretrained on approximately 1.15B tokens. Results should be interpreted in the context of its size, training budget, and objectives.


In [ ]:
from datasets import load_dataset
import re
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
from tqdm.auto import tqdm
import numpy as np
import time

In [ ]:

DATASET_ID = "Rowan/hellaswag"
SEED = 1337
SAMPLE_SIZE = 200

dataset = load_dataset(
    DATASET_ID,
    split="validation",
)

eval_dataset = dataset

print("=" * 60)
print("NEOLM — HELLASWAG BENCHMARK")
print("=" * 60)
print(f"Dataset:          {DATASET_ID}")
print(f"Validation size:  {len(dataset):,}")
print(f"Evaluation size:  {len(eval_dataset):,}")
print(f"Choices/example:  4")
print(f"Random baseline:  25%")
print("=" * 60)

sample = eval_dataset[0]

print("\nCONTEXT:")
print(sample["ctx"])

print("\nCANDIDATE ENDINGS:")
for i, ending in enumerate(sample["endings"]):
    print(f"{i}: {ending}")

print("\nCorrect choice:", sample["label"])

In [ ]:
MODEL_DIR = "../artifacts/model"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    local_files_only=True,
).to(device).eval()


def preprocess(text):
    text = text.strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    return text.replace("  ", " ")


def prepare_example(row):
    context = row["ctx_a"] + " " + row["ctx_b"].capitalize()

    return {
        "context": preprocess(
            row["activity_label"] + ": " + context
        ),
        "endings": [
            preprocess(ending)
            for ending in row["endings"]
        ],
        "label": int(row["label"]),
    }


sample = prepare_example(eval_dataset[0])

print("=" * 60)
print("NEOLM — HELLASWAG PREPROCESSING")
print("=" * 60)
print("Context:", sample["context"])

for i, ending in enumerate(sample["endings"]):
    print(f"\nChoice {i}: {ending}")

print("\nCorrect choice:", sample["label"])
print("Model parameters:", f"{model.num_parameters():,}")
print("=" * 60)

In [ ]:
@torch.inference_mode()
def score_ending(context, ending):
    context_ids = tokenizer.encode(
        context,
        add_special_tokens=False,
    )

    ending_ids = tokenizer.encode(
        " " + ending.strip(),
        add_special_tokens=False,
    )

    max_length = model.config.max_position_embeddings

    assert context_ids and ending_ids
    assert len(ending_ids) < max_length

    # Preserve the ending and truncate context if needed
    context_ids = context_ids[
        -(max_length - len(ending_ids)):
    ]

    input_ids = torch.tensor(
        [context_ids + ending_ids],
        dtype=torch.long,
        device=model.device,
    )

    logits = model(
        input_ids=input_ids,
        use_cache=False,
    ).logits[0]

    start = len(context_ids) - 1
    end = start + len(ending_ids)

    log_probs = F.log_softmax(
        logits[start:end].float(),
        dim=-1,
    )

    targets = torch.tensor(
        ending_ids,
        device=model.device,
    )

    token_scores = log_probs.gather(
        1, targets.unsqueeze(1)
    ).squeeze(1)

    return {
        "loglikelihood": token_scores.sum().item(),
        "normalized": token_scores.mean().item(),
    }


# Evaluate one example
sample = prepare_example(eval_dataset[0])

scores = [
    score_ending(sample["context"], ending)
    for ending in sample["endings"]
]

raw_prediction = max(
    range(4),
    key=lambda i: scores[i]["loglikelihood"],
)

normalized_prediction = max(
    range(4),
    key=lambda i: scores[i]["normalized"],
)

print("=" * 60)
print("NEOLM — HELLASWAG SINGLE EXAMPLE")
print("=" * 60)

for i, score in enumerate(scores):
    print(
        f"Choice {i} | "
        f"Log-Likelihood: {score['loglikelihood']:8.2f} | "
        f"Normalized: {score['normalized']:.4f}"
    )

print("-" * 60)
print(f"Correct answer:       {sample['label']}")
print(f"Raw prediction:       {raw_prediction}")
print(f"Normalized prediction:{normalized_prediction:>3}")
print(f"Raw correct:          {raw_prediction == sample['label']}")
print(f"Normalized correct:   {normalized_prediction == sample['label']}")
print("=" * 60)

In [ ]:
raw_correct = 0
normalized_correct = 0
predictions = []

start_time = time.perf_counter()

for row in tqdm(eval_dataset, desc="Evaluating HellaSwag"):
    example = prepare_example(row)

    scores = [
        score_ending(example["context"], ending)
        for ending in example["endings"]
    ]

    raw_pred = int(np.argmax([
        s["loglikelihood"] for s in scores
    ]))

    norm_pred = int(np.argmax([
        s["normalized"] for s in scores
    ]))

    label = example["label"]

    raw_correct += raw_pred == label
    normalized_correct += norm_pred == label

    predictions.append({
        "label": label,
        "raw_prediction": raw_pred,
        "normalized_prediction": norm_pred,
    })

elapsed = time.perf_counter() - start_time
n = len(predictions)

print(f"\n{'=' * 60}")
print("NEOLM — HELLASWAG PILOT RESULTS")
print(f"{'=' * 60}")
print(f"Examples:             {n:,}")
print(f"Random baseline:      25.00%")
print(f"Raw accuracy:         {raw_correct / n:.2%}")
print(f"Normalized accuracy:  {normalized_correct / n:.2%}")
print(f"Raw correct:          {raw_correct}/{n}")
print(f"Normalized correct:   {normalized_correct}/{n}")
print(f"Runtime:              {elapsed:.1f}s")
print(f"{'=' * 60}")

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

results = {
    "model": "MaagDeveloper/NeoLM-Small-Base",
    "benchmark": "hellaswag",
    "dataset": DATASET_ID,
    "evaluation": "custom_zero_shot",
    "examples": n,
    "random_baseline": 0.25,
    "raw_accuracy": raw_correct / n,
    "normalized_accuracy": normalized_correct / n,
    "raw_correct": raw_correct,
    "normalized_correct": normalized_correct,
    "runtime_seconds": round(elapsed, 2),
}

output_file = RESULTS_DIR / "neolm_hellaswag.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print("=" * 60)
print("NEOLM — HELLASWAG RESULTS SAVED")
print("=" * 60)
print(f"Examples:            {n:,}")
print(f"Raw Accuracy:        {results['raw_accuracy']:.2%}")
print(f"Normalized Accuracy: {results['normalized_accuracy']:.2%}")
print(f"Saved to:            {output_file}")
print("=" * 60)


## Results & Conclusion

Evaluated NeoLM-Small-Base on the full HellaSwag validation split (10,042 examples).

| Metric | Accuracy |
|---|---:|
| Random baseline | 25.00% |
| Raw accuracy | 28.03% |
| Length-normalized accuracy | 29.37% |

### Findings

- NeoLM performs modestly above random chance.
- Length normalization produces slightly better accuracy.
- Commonsense continuation remains a limitation of the released 61M model.
- These results do not establish the performance ceiling of the NeoLM architecture.

### Limitations

- Results use a custom zero-shot scoring implementation.
- Direct comparison with official leaderboard results requires matching the reference evaluation protocol.
- The model was pretrained on approximately 1.15B tokens without instruction fine-tuning.
